# Tutorial 08 -- Dispersive Shift and Dressed Frequencies

Connect the low-level Hamiltonian, the dressed energy spectrum, and the photon-number-dependent qubit transition frequencies in one place.

**Prerequisites.** Tutorials 01, 02, and 07 are recommended first.


## 1. Goal

We will compare dressed energies from exact diagonalization with the manifold transition helpers exposed by the public API.


## 2. Physical Background

The dispersive approximation is easiest to trust when the dressed energy picture and the transition helper functions tell the same story. This notebook keeps both views side by side.


## 3. Imports


In [ ]:
from __future__ import annotations

from functools import partial
from pathlib import Path
import sys

REPO_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "pyproject.toml").exists() and (candidate / "cqed_sim").is_dir()
    ),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Could not resolve the repository root from the notebook working directory.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import qutip as qt

from cqed_sim import (
    AmplifierChain,
    BosonicModeSpec,
    DispersiveCouplingSpec,
    DispersiveReadoutTransmonStorageModel,
    DispersiveTransmonCavityModel,
    DisplacementGate,
    FrameSpec,
    NoiseSpec,
    Pulse,
    PurcellFilter,
    QubitMeasurementSpec,
    ReadoutChain,
    ReadoutResonator,
    RotationGate,
    SidebandDriveSpec,
    SequenceCompiler,
    SimulationConfig,
    StatePreparationSpec,
    TransmonModeSpec,
    UniversalCQEDModel,
    build_displacement_pulse,
    build_rotation_pulse,
    build_sideband_pulse,
    carrier_for_transition_frequency,
    coherent_state,
    compute_energy_spectrum,
    fock_state,
    manifold_transition_frequency,
    measure_qubit,
    prepare_simulation,
    prepare_state,
    pure_dephasing_time_from_t1_t2,
    qubit_state,
    run_rabi,
    run_ramsey,
    run_spectroscopy,
    run_t1,
    run_t2_echo,
    sideband_transition_frequency,
    simulate_batch,
    simulate_sequence,
)
from cqed_sim.plotting import plot_energy_levels
from cqed_sim.pulses import gaussian_envelope, square_envelope
from cqed_sim.sim import (
    cavity_wigner,
    conditioned_bloch_xyz,
    mode_moments,
    qubit_conditioned_mode_moments,
    readout_response_by_qubit_state,
    reduced_cavity_state,
    reduced_qubit_state,
    reduced_storage_state,
    storage_photon_number,
    subsystem_level_population,
    transmon_level_populations,
)
from tutorials.tutorial_support import (
    GHz,
    MHz,
    angular_to_ghz,
    angular_to_hz,
    angular_to_mhz,
    final_expectation,
    fit_echo_signal,
    fit_exponential_decay,
    fit_lorentzian_peak,
    fit_rabi_vs_amplitude,
    fit_rabi_vs_duration,
    fit_ramsey_signal,
    ns,
    us,
)

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (7.0, 4.2)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False


## 4. Simulation Parameters


In [ ]:
max_n = 6


## 5. Model Construction


In [ ]:
model = DispersiveTransmonCavityModel(
    omega_c=GHz(5.05),
    omega_q=GHz(6.25),
    alpha=MHz(-220.0),
    chi=MHz(-2.6),
    kerr=MHz(-0.002),
    n_cav=10,
    n_tr=3,
)
frame = FrameSpec(omega_c_frame=model.omega_c, omega_q_frame=model.omega_q)
lab_spectrum = compute_energy_spectrum(model, frame=FrameSpec(), levels=14)
rot_spectrum = compute_energy_spectrum(model, frame=frame, levels=14)


## 6. Pulse / Sequence Construction


In [ ]:
n_values = np.arange(max_n + 1)
transition_mhz = np.array([angular_to_mhz(manifold_transition_frequency(model, int(n), frame=frame)) for n in n_values])
line_spacing_mhz = np.diff(transition_mhz)


## 7. Running the Simulation


In [ ]:
print("Transition frequencies [MHz] in the matched rotating frame:")
for n, value in zip(n_values, transition_mhz, strict=True):
    print(f"  n = {n}: {value:+.4f} MHz")
print("Adjacent line spacing [MHz]:", np.round(line_spacing_mhz, 4))


## 8. Visualizing the Results


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.2))
plot_energy_levels(lab_spectrum, max_levels=12, energy_scale=1.0 / (2.0 * np.pi * 1.0e6), energy_unit_label="MHz", title="Lab-frame dressed energies", ax=axes[0])
axes[1].plot(n_values, transition_mhz, "o-")
axes[1].set_xlabel("Photon number n")
axes[1].set_ylabel(r"$\omega_{ge}(n) / 2\pi$ [MHz]")
axes[1].set_title("Dressed qubit transition versus bosonic occupation")
plt.show()


## 9. Physical Interpretation

The energy-level plot is the dressed-spectrum view, while the `omega_ge(n)` curve is the spectroscopy view. Both are generated from the same public model object, so agreement between them is a strong sanity check before moving into calibration notebooks.


## 10. Exercises / Next Steps

- Increase the cavity Kerr and see when the transition-versus-`n` curve bends away from a strictly linear `chi` shift.
- Compare the rotating-frame and lab-frame spectra directly by changing `frame` in the `compute_energy_spectrum(...)` calls.
- Continue to Tutorials 09 and 10 for Rabi calibrations.
